# Interpret checkpoint (single episode)

Load a **``.pt``** checkpoint and a snapshot **JSON**, build :class:`~env.game_simulator.GameSimulator`, then roll out **one** episode with the policy (same logic as ``env/interpret_model.py``).

**Flow:** ``GameSimulator.from_json`` → ``print_info_snapshot(deck="summary")`` → loop: policy → ``step_and_print(act, deck="summary")`` (validates, steps, prints full snapshot).

**One environment only:** there is a single `BalatroEnv` behind `GameSimulator`—no `VectorEnv`, no parallel workers, and no multi-rollout batch. Evaluation is always one trajectory until terminal or `MAX_STEPS`.

**Colab:** the next cell mounts Google Drive. Set ``REPO_DIR`` to your clone path (default matches ``TrainCombat.ipynb``). You can also set env ``BALATRO_AGENT_REPO`` before running that cell to skip editing.

**Local:** the mount step is skipped; set ``REPO_DIR`` to your local repo root.


In [1]:
import os
import sys

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass  # local: set REPO_DIR below

REPO_DIR = os.environ.get(
    "BALATRO_AGENT_REPO",
    "/content/drive/MyDrive/balatro-agent-project",
)
REPO_ROOT = REPO_DIR

os.chdir(REPO_DIR)

for p in (
    REPO_DIR,
    os.path.join(REPO_DIR, "balatro_lite_gym"),
):
    if p not in sys.path:
        sys.path.insert(0, p)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

import numpy as np
import torch
from torch.distributions import Categorical

from agent.model import CombatPPOAgent
from agent.ppo import get_card_mask, mask_logits
from agent.ppo_config import PPOConfig
from defs import CARD_RANK_LABELS, CardRank
from env.game_simulator import GameSimulator
from env.lite_combat_env import adapt_lite_vector_obs, dict_to_tensors
from environment import MAX_HAND_LENGTH
from util import rank_from_card_id, suit_from_card_id


In [3]:
_SUIT_GLYPHS = "♠♥♦♣"


def _nested_obs_add_batch_dim(obs_raw: dict) -> dict:
    out: dict = {}
    for k, v in obs_raw.items():
        if isinstance(v, dict):
            out[k] = {
                kk: np.asarray(vv, dtype=np.float32)[np.newaxis, ...]
                for kk, vv in v.items()
            }
        else:
            out[k] = np.asarray(v, dtype=np.float32)[np.newaxis, ...]
    return out


def _card_face(card_id: int) -> str:
    r = CardRank(rank_from_card_id(card_id))
    s = int(suit_from_card_id(card_id))
    return f"{CARD_RANK_LABELS[r]}{_SUIT_GLYPHS[s]}"


def _action_from_checkpoint(
    agent: CombatPPOAgent,
    obs_t: dict,
    *,
    stochastic: bool,
) -> tuple[np.ndarray, float]:
    with torch.no_grad():
        sel_logits, exec_logits, value = agent(obs_t)
    card_mask = get_card_mask(obs_t)
    sel_logits = mask_logits(sel_logits, card_mask)

    if stochastic:
        sel_dist = Categorical(logits=sel_logits)
        exec_dist = Categorical(logits=exec_logits)
        card_sels = sel_dist.sample()
        executions = exec_dist.sample()
    else:
        card_sels = sel_logits.argmax(dim=-1)
        executions = exec_logits.argmax(dim=-1)

    exec_bit = int(executions[0].item())
    action = np.concatenate(
        [card_sels[0].cpu().numpy().astype(np.int8), np.array([exec_bit], dtype=np.int8)],
        axis=0,
    )
    v = float(value[0].squeeze(-1).item())
    return action, v


In [6]:
# --- edit paths / options ---
CKPT_PATH = Path(REPO_ROOT) / "checkpoints" / "combat_ppo_iter_400.pt"
SNAPSHOT_PATH = Path(REPO_ROOT) / "snapshots" / "snapshot_no_jokers_level1.json"
SEED = 0
STOCHASTIC = False
MAX_STEPS = 10_000
DEVICE = None  # None -> cuda if available else cpu

ckpt_path = CKPT_PATH.resolve()
snap_path = SNAPSHOT_PATH.resolve()
assert ckpt_path.is_file(), f"checkpoint not found: {ckpt_path}"
assert snap_path.is_file(), f"snapshot not found: {snap_path}"

device = torch.device(
    DEVICE if DEVICE is not None else ("cuda" if torch.cuda.is_available() else "cpu")
)


In [7]:
try:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(ckpt_path, map_location=device)

cfg = ckpt.get("config")
if cfg is None:
    cfg = PPOConfig()
    print("warning: checkpoint has no 'config'; using default PPOConfig()", file=sys.stderr)

agent = CombatPPOAgent(
    d_model=cfg.d_model,
    nhead=cfg.nhead,
    dim_ff=cfg.dim_ff,
    dropout=cfg.dropout,
).to(device)
agent.load_state_dict(ckpt["model_state_dict"])
agent.eval()

sim = GameSimulator.from_json(snap_path, seed=SEED)
sim.print_info_snapshot(deck="summary")
obs_raw = sim.obs
info = sim.info

step = 0
print(f"checkpoint: {ckpt_path}")
print(f"snapshot:   {snap_path}")
print(f"device:     {device}  |  action: {'sample' if STOCHASTIC else 'argmax'}")

while step < MAX_STEPS:
    snap = info["snapshot"]
    print(f"\n=== Step {step} (before action) ===")

    obs_t = dict_to_tensors(
        adapt_lite_vector_obs(_nested_obs_add_batch_dim(obs_raw)),
        device,
    )
    action, val = _action_from_checkpoint(agent, obs_t, stochastic=STOCHASTIC)

    indices = [i for i in range(MAX_HAND_LENGTH) if int(action[i]) == 1]
    n_real = len(snap.hand)
    indices = [i for i in indices if i < n_real]
    exec_bit = int(action[MAX_HAND_LENGTH])
    verb = "PLAY" if exec_bit == 0 else "DISCARD"
    picked = [_card_face(snap.hand[i].card_id) for i in indices if i < len(snap.hand)]
    print(
        f"  policy value: {val:+.4f}  |  {verb}  |  selected slots: {indices}  ->  {picked}"
    )

    card_sel = action[:MAX_HAND_LENGTH].astype(np.int8)
    execution = int(action[MAX_HAND_LENGTH])
    action_type = 1 if execution == 0 else 0
    act = {"selection": card_sel, "action_type": int(action_type)}

    outcome = sim.step_and_print(act, deck="summary")
    obs_raw, info = sim.obs, sim.info
    step += 1

    if outcome.stepped:
        r = float(outcome.reward)
        term = outcome.terminated
        print(f"  env reward: {r:+.4f}  |  terminated={term}")
        if r < 0 and not term:
            print(
                "  (invalid selection or illegal discard; env returned -1 without ending)"
            )
        if term:
            print(
                "(hint) Episode over: win."
                if outcome.combat_won
                else "(hint) Episode over: loss."
            )
            break
    else:
        print("  env reward: (no step — invalid action)  |  terminated=False")
else:
    print(f"\nStopped after MAX_STEPS={MAX_STEPS} without terminal.")



=== Blind / score ===
  round chips: 0 / target 300  (need 300 more)

=== Resources ===
  hand_size=8  hands_left=4  discards_left=3

=== Boss ===
  blind_id=-1  (non-boss blind (Small/Big))

=== Hand ===
              │ 4♥ │ J♦ │ 2♥ │ Q♥ │ T♦ │ A♦ │ 4♦ │ T♠
  ────────────┼────┼────┼────┼────┼────┼────┼────┼────
  slot        │ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ 7 
  card        │ 4♥ │ J♦ │ 2♥ │ Q♥ │ T♦ │ A♦ │ 4♦ │ T♠
  enhancement │    │    │    │    │    │    │    │   
  edition     │    │    │    │    │    │    │    │   
  debuff      │    │    │    │    │    │    │    │   

=== Jokers ===
  (none)

=== Draw pile ===
  cards in draw pile: 44
  ┌───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬───┐
  │   │ A │ 2 │ 3 │ 4 │ 5 │ 6 │ 7 │ 8 │ 9 │ T │ J │ Q │ K │
  ├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
  │ ♣ │ 1 │ 1 │ 1 │ 1 │ 1 │ 1 │ 1 │ 1 │ 1 │ 1 │ 1 │ 1 │ 1 │
  │ ♦ │   │ 1 │ 1 │   │ 1 │ 1 │ 1 │ 1 │ 1 │   │   │ 1 │ 1 │
  │ ♥ │ 1 │   │ 1 │   │ 1 │ 1 │ 1 │ 1 │ 1 │ 1 

KeyboardInterrupt: 